# Cost Optimization

*Author: [Sai Guntupalli](https://github.com/sai-guntupalli) · [sai-guntupalli/data-engineering-handbook](https://github.com/sai-guntupalli/data-engineering-handbook)*



*How to use this notebook.* Read the explanation cells and run the code cells below them. Python examples list their imports, so install what a section needs before running it. SQL and shell snippets are illustrative: they show the syntax for the platform named in that section and need that platform (or a compatible engine) to run.

Cost optimization in data engineering is delivering the same SLAs and business value with fewer resources. It is not a one-off project: storage grows, pipelines multiply, and query patterns drift, so cost work is a recurring discipline alongside performance and reliability.

Platform spend falls into four buckets: **storage**, **compute** (warehouse credits, cluster hours, serverless bytes scanned), **data movement** (egress, cross-region replication), and **operational overhead** (backfills, retries, duplicate pipelines). Interviewers want to see you find the biggest bucket with data, not guess.

## Why is cost optimization a core data engineering skill?

Data platforms can scale costs linearly or worse with data volume and usage. Cost optimization ensures the platform remains sustainable while meeting SLAs. It requires understanding storage layout, compute behavior, query patterns, and operational practices like backfills and compaction.

Frame it as **engineering with a second objective function**. A pipeline that meets its SLA while scanning 50 TB for 20 rows of output is a defect. The efficient patterns - partition pruning, columnar formats, incremental processing, right-sized partitions - are usually the same ones that improve performance and reliability, so cost work rarely forces a trade-off with quality.

Start with measurement, not guesses. The warehouse's own billing views explain most of a bill:

```sql
-- BigQuery: the 20 most expensive queries in the last 7 days
SELECT
    query,
    user_email,
    total_bytes_billed / POW(1024, 4) AS tib_billed,
    ROUND(total_slot_ms / 1000 / 3600, 2) AS slot_hours,
    creation_time
FROM `region-us`.INFORMATION_SCHEMA.JOBS_BY_PROJECT
WHERE creation_time >= TIMESTAMP_SUB(CURRENT_TIMESTAMP(), INTERVAL 7 DAY)
  AND job_type = 'QUERY' AND state = 'DONE'
ORDER BY total_bytes_billed DESC
LIMIT 20;
```

```sql
-- Snowflake: credit consumption per warehouse per day
SELECT warehouse_name, DATE(start_time) AS usage_date,
       SUM(credits_used) AS credits
FROM SNOWFLAKE.ACCOUNT_USAGE.WAREHOUSE_METERING_HISTORY
WHERE start_time >= DATEADD('day', -30, CURRENT_TIMESTAMP())
GROUP BY warehouse_name, usage_date
ORDER BY credits DESC;
```

Two or three queries usually explain most of the bill: a handful of dashboards scanning raw tables, a backfill re-reading history, and an oversized always-on warehouse. Find those first.

**The cost model has five layers, and you fix the biggest marginal one first.** Storage is cheap per GB-month but grows forever. Compute is billed by time or by bytes scanned and dominates most bills. Requests and egress are billed per API call and per GB moved between regions. Orchestration and observability are billed per task-minute, per monitored metric, and per log byte. People are the largest cost in a platform nobody can operate. A typical bill is 60-70 percent compute, 10-20 percent storage, and the rest spread across the other three.

**Track unit economics, not just totals.** Useful units are cost per query, cost per pipeline run, cost per dataset-month, and cost per team or product. Totals can fall while a unit cost rises: spend drops because traffic dropped, but each query still scans 40 TB. In an interview, name the unit you would manage, not only the total you would cut.

**Worked example.** A platform spends 50,000 a month: compute 30,500, storage 7,500, egress 4,000, orchestration 3,000, observability and people 5,000. Two dashboards scanning raw events account for 12,000 of compute. Partitioning them cuts 9,000, an 18 percent bill reduction from one change and far more than a discount would give.

**Common mistake.** Negotiating a discount before fixing layout. A 20 percent discount on a workload that scans three times more than it needs still loses money, and a one-year commitment locks the waste in.

## How do you reduce warehouse and lake query costs?

Start by reducing data scanned: use columnar formats, partition pruning, clustering/sort keys, and selective projections. Then reduce repeated compute by materializing expensive shared transformations. Finally, set workload controls such as query limits, warehouse auto-suspend, reservations, and alerts.

| Lever | Typical saving | Effort |
|---|---|---|
| Stop scanning unused columns | 30-95% of bytes | low |
| Partition pruning on filter columns | 50-99% of bytes | medium |
| Cluster/sort within partitions | 20-70% of bytes | medium |
| Materialize repeated joins/aggregations | 40-90% of compute | medium-high |
| Auto-suspend and right-size warehouses | 20-60% of credits | low |
| Query guardrails and quotas | bounds the tail | low-medium |

```sql
-- Expensive: every column, all history, filter applied late
SELECT * FROM raw.events_json WHERE event_date >= DATE '2026-10-01';

-- Cheap: three columns, partition-key filter pushed down
SELECT event_date, user_id, COUNT(*) AS events
FROM curated.events
WHERE event_date >= DATE '2026-10-01' AND event_date < DATE '2026-10-08'
GROUP BY event_date, user_id;
```

```bash
# BigQuery: estimate bytes scanned without executing the query
bq query --dry_run --use_legacy_sql=false \
  'SELECT event_date, COUNT(*) FROM `acme.curated.events`
   WHERE event_date = DATE "2026-10-04" GROUP BY event_date'
```

On Snowflake, inspect the query profile and its partition-pruning percentage before and after a change.

**Failure mode:** micro-optimizing a cheap query while one dashboard scans a raw table every 15 minutes. Rank by total spend (frequency x unit cost), not single-query cost.

**The billing model decides which lever works.** Scan-billed engines (BigQuery on-demand, Athena, Snowflake serverless features) charge for bytes read after pruning, so the lever is layout: partitions, clustering, column pruning, and pre-aggregation. Time-billed engines (Snowflake credits, BigQuery slots, Spark node-hours) charge for wall-clock, so the lever is concurrency, warehouse size, and idle time. A change can help one and hurt the other: a large warehouse finishes faster but bills more per second.

**Common mistake.** Assuming a repeated query is free. Result caches expire, and on a scan-billed engine a cache miss re-bills the full scan. Point dashboards at materialized tables, not raw ones.

In [ ]:
# Review only: model the saving from partition plus column pruning
RAW_BYTES = 40 * 1024 ** 4          # 40 TiB scanned with no pruning
PARTITION_SELECTIVITY = 7 / 365     # a 7-day window over a year of data
COLUMN_FRACTION = 0.12              # 3 of 25 columns are actually needed
PRICE_PER_TIB = 6.25

def scan_cost(bytes_scanned):
    return bytes_scanned / 1024 ** 4 * PRICE_PER_TIB

before = RAW_BYTES
after = RAW_BYTES * PARTITION_SELECTIVITY * COLUMN_FRACTION
print(f"before {before / 1024 ** 4:6.1f} TiB  {scan_cost(before):9.2f}")
print(f"after  {after / 1024 ** 4:6.1f} TiB  {scan_cost(after):9.2f}")

## How do you explain cost trade-offs in a data architecture interview?

Frame cost as a product of storage, compute, scan volume, orchestration frequency, and operational complexity. A cheap pipeline that is impossible to debug is not truly cheap. A more expensive serving table may be worth it if it prevents hundreds of dashboard queries from recomputing the same logic.

Example trade-off:

+ raw-only lake: cheap storage, expensive repeated queries
+ curated marts: extra storage and ETL, cheaper and faster BI
+ streaming pipeline: low latency, higher operational cost
+ daily batch: lower cost, higher latency

Quantify it. If 200 analysts run 30 ad-hoc queries a day over a 40 TB raw table, but the pre-aggregated mart is 2 TB:

| Option | Scan per query | Daily scan | Notes |
|---|---|---|---|
| Raw table | ~35 TB | ~210 PB/day | unusable, dominates the bill |
| Pre-aggregated mart | ~40 GB | ~240 TB/day | ~99.9% less scan |
| Mart build cost | - | one 6 TB scan/day | ~3% overhead |

The answer in an interview is a **breakeven statement**: "materializing the mart costs one full scan a day and saves about 200 full scans, so it pays back within hours as long as the mart is reused a few times per day."

Do not optimize away observability (a bad publish costs more than the check), isolation (mixing a backfill with interactive BI hurts everyone), or reproducibility (deleting raw history makes incident recovery impossible - tier it to cheap object storage instead).

**Breakeven arithmetic for a serving layer.** Materializing a mart pays when `refresh_cost < reuses x scan_saved`. Put real numbers on both sides and the argument stops being a matter of taste. A mart that costs one 6 TB scan a day and saves 200 full scans of a 40 TB table pays back within hours.

**Common mistake.** Optimizing for the cheapest option per query instead of the lowest total cost of ownership. The mart is not free, but a page that loads in two seconds and stops 200 raw scans is usually the cheaper design.

```sql
-- Cost per dashboard query: on-demand pricing is linear in bytes billed
SELECT
    REGEXP_EXTRACT(query, r'FROM\s+`([^`]+)`') AS source_table,
    COUNT(*) AS runs,
    ROUND(SUM(total_bytes_billed) / POW(1024, 4), 1) AS tib_billed,
    ROUND(SUM(total_bytes_billed) / POW(1024, 4) * 6.25, 2) AS est_cost_usd
FROM `region-us`.INFORMATION_SCHEMA.JOBS_BY_PROJECT
WHERE creation_time >= TIMESTAMP_SUB(CURRENT_TIMESTAMP(), INTERVAL 30 DAY)
  AND job_type = 'QUERY' AND state = 'DONE'
GROUP BY source_table
ORDER BY tib_billed DESC
LIMIT 20;
```

## What are the main cost drivers in data platforms?

Typical drivers include:

+ compute time (clusters, warehouses, serverless slots)
+ bytes scanned and shuffles
+ storage growth (raw + derived + duplicates)
+ data movement (egress, cross-region transfers)
+ operational overhead (frequent backfills, retries)

| Driver | Metric | Typical control |
|---|---|---|
| Warehouse compute | credits or slot-hours | auto-suspend, right-size, budgets |
| Serverless scan | bytes billed | partitioning, clustering, dry runs |
| Spark/cluster compute | DBU or node-hours | right-size, spot, adaptive execution |
| Storage | TB-month by layer | lifecycle to cold tiers, compaction |
| Data movement | GB egress | co-locate compute and storage |
| Orchestration | task-minutes, retries | fewer, larger tasks; idempotent reruns |

Time travel and failsafe are easy to forget because they are billed storage:

```sql
-- Snowflake: storage by schema, separating active from time-travel bytes
SELECT table_schema, table_name,
       ROUND(active_bytes / POW(1024, 3), 2)      AS active_gb,
       ROUND(time_travel_bytes / POW(1024, 3), 2) AS time_travel_gb,
       ROUND(failsafe_bytes / POW(1024, 3), 2)    AS failsafe_gb
FROM SNOWFLAKE.ACCOUNT_USAGE.TABLE_STORAGE_METRICS
WHERE active_bytes > 100 * POW(1024, 3)
ORDER BY active_bytes DESC
LIMIT 50;
```

If a table retains 10 TB of active data and 40 TB of time travel, reducing retention is a direct cost fix that no query tuning will find.

**Lifecycle tiering and retention are pure storage levers.** Object stores price by GB-month and access pattern, and hot, infrequent, and archive tiers differ by roughly 5-10x in rate, at the cost of retrieval latency and minimum-storage-duration charges. Retention settings (time travel days, snapshot expiry, raw history) decide how much is billable at all. Tiering is safe for immutable data that is read rarely, and a trap for anything that a forgotten job reads every hour.

**Worked example.** 200 TB of raw events: 80 TB is read within 90 days and stays hot at 0.023 per GB-month, 60 TB moves to an infrequent tier at 0.0125, and 60 TB of that is eligible for archive at 0.004. Monthly cost is 1,884 + 768 + 246 = 2,898, against 4,710 if everything stayed hot. That is a 38 percent storage reduction before retrieval charges.

**Common mistake.** Deleting raw history or disabling time travel to save storage, then losing the ability to replay a bad publish. Tier it instead, and keep the retention window your recovery plan actually needs.

## How does partitioning affect cost and performance?

Good partitioning reduces scanned data by enabling partition pruning. Bad partitioning creates too many small partitions, increasing metadata overhead and small files. Partitioning should match common query filters and data distribution, and be reevaluated as workloads evolve.

Guidelines: partition on the columns that appear in `WHERE` clauses, usually a date; keep cardinality moderate (thousands, not hundreds of thousands); do not partition on high-cardinality ids; and recheck that filters do not wrap the partition column in a function.

```sql
-- A function on the partition column defeats pruning
SELECT COUNT(*) FROM curated.events WHERE DATE(event_ts) = DATE '2026-10-04';

-- Pruning-friendly: filter the raw partition column directly
SELECT COUNT(*) FROM curated.events
WHERE event_ts >= TIMESTAMP '2026-10-04 00:00:00'
  AND event_ts <  TIMESTAMP '2026-10-05 00:00:00';
```

```sql
-- Confirm the result: actual bytes processed and slot time
SELECT total_bytes_processed / POW(1024, 3) AS gb_processed,
       total_slot_ms / 1000 / 3600 AS slot_hours
FROM `region-us`.INFORMATION_SCHEMA.JOBS_BY_PROJECT
WHERE job_id = 'job_abc123';
```

Partition **and cluster**: partitioning prunes partitions, while clustering sorts within them so row-group and micro-partition pruning can skip more. On Snowflake that is `CLUSTER BY`, on BigQuery a cluster key, on Iceberg a sort order. Revisit the scheme every few months, because a stale partition choice quietly becomes a full-scan generator.

**Pruning is a pricing control, not only a speed control.** The planner compares the filter to partition metadata and skips whole partitions before reading data. Over-partitioning gives that back: tens of thousands of tiny partitions mean more metadata reads, more file opens, and more storage requests, each of which is billed or slows planning. The right granularity is the coarsest one that still prunes your dominant filter.

```sql
-- Find partitions that are far larger than the rest, or suspiciously empty
SELECT table_name, partition_id, total_rows, last_modified_time
FROM `region-us`.INFORMATION_SCHEMA.PARTITIONS
WHERE table_name = 'events'
ORDER BY total_rows DESC
LIMIT 20;
```

**Common mistake.** Partitioning on ingestion time instead of event time. Late-arriving rows land in the wrong partition, and every event-time filter then scans the whole table.

## What is the small files problem and why does it increase cost?

Small files increase scheduling and metadata overhead and reduce scan efficiency. Engines spend more time opening and planning files than processing data. Small files often appear from streaming writes or overly granular partitions and typically require compaction or clustering to fix.

The cost is threefold: **metadata/planning** (each file is an open plus a footer parse), **compute** (tiny tasks waste slot time), and **storage** (per-object overhead). A partition with 50,000 one-MB files can be slower than the same data in 50 files of 1 GB.

```sql
-- Iceberg: transactional compaction instead of hand-rewriting files
CALL catalog.system.rewrite_data_files(
    table => 'curated.orders',
    options => map('target-file-size-bytes', '268435456')
);
```

```bash
# Count and size files in a prefix to spot a small-files partition
aws s3api list-objects-v2 --bucket acme-curated --prefix orders/ds=2026-10-04/ \
  --query 'Contents[].Size' --output text | tr '\t' '\n' | sort -n | tail -5
```

**Failure mode:** compacting without controlling the writer. If a streaming job produces one-second micro-batches, compaction runs forever and costs more than it saves. Increase the trigger interval or buffer first, then compact history. Iceberg and Delta make this safe by rewriting files under a transaction; doing it by hand on plain Parquet can corrupt readers mid-swap.

**Metadata and request charges are the hidden bill.** Every file is at least one object-store request and one footer parse. At 300,000 files per day, a scan pays hundreds of thousands of GET requests on top of engine planning time, and object stores bill those requests separately from stored bytes. Compaction trades a bounded rewrite now for cheaper reads and fewer requests later, so the question is how many scans it takes to pay back.

In [ ]:
# Review only: does a compaction rewrite pay for itself?
FILES = 300_000
AVG_MB = 3
TARGET_MB = 256
REWRITE_COST_PER_GB = 0.005
SCANS_PER_DAY = 40
REQUEST_COST_PER_1K = 0.0004

gb = FILES * AVG_MB / 1024
rewrite_cost = gb * REWRITE_COST_PER_GB
requests_after = gb * 1024 / TARGET_MB
daily_request_saving = (FILES - requests_after) / 1000 * REQUEST_COST_PER_1K * SCANS_PER_DAY
print(f"data {gb:.0f} GB, one-time rewrite cost {rewrite_cost:.2f}")
print(f"daily request saving {daily_request_saving:.2f}")
print(f"payback days {rewrite_cost / daily_request_saving:.2f}" if daily_request_saving else "no saving")

## How do you choose a target file size for Parquet tables?

You choose a size that balances parallelism and overhead. Too small increases file count and planning cost; too large reduces parallelism and can slow selective queries. Many teams target hundreds of MB per file, but the correct value depends on the engine, storage, and query patterns.

A practical starting point:

+ **Batch lake tables:** 128-512 MB per file, up to 1 GB for append-only facts.
+ **Small dimension tables:** a single file is fine.
+ **Wide/nested tables:** row groups around 128 MB, files under ~1 GB.
+ **Streaming with frequent commits:** a few MB per file, compacted more often.

The math matters more than the number: aim for at least one file per CPU core you want to parallelize across, and at most a few thousand files per query. For a 500 GB partition on a 200-core cluster, 200-500 files of 1-2.5 GB is reasonable; 30 files of 16 GB serializes work, and 200,000 files of 2.5 MB wastes the cluster on planning.

In [ ]:
# ILLUSTRATIVE - requires pyspark: install to run, or read on.
# PySpark: size the write deliberately instead of inheriting the shuffle count
from pyspark.sql import SparkSession

spark = SparkSession.builder.appName("right-size-write").getOrCreate()
TARGET_BYTES = 256 * 1024 * 1024
data = spark.read.parquet("s3://acme-curated/orders/")
total_bytes = 900 * 1024 ** 3                 # from S3 inventory / table metadata
files = max(1, total_bytes // TARGET_BYTES)

(
    data.repartition(int(files))
    .write.mode("overwrite")
    .option("parquet.block.size", str(128 * 1024 * 1024))
    .parquet("s3://acme-curated/orders_compact/")
)

spark.conf.set("spark.sql.files.openCostInBytes", str(TARGET_BYTES))
spark.conf.set("spark.sql.files.maxPartitionBytes", str(TARGET_BYTES))

`spark.sql.files.openCostInBytes` tells Spark how expensive opening a file is relative to scanning bytes; raising it makes the planner pack more files per task and avoids one tiny task per file.

**Size for the query, not for the writer.** Target roughly one file per core you want busy, then round to a size that keeps row-group statistics selective. A 500 GB partition on a 200-core cluster wants 200-500 files. The same partition in 30 files serializes work, and in 200,000 files the cluster spends more time planning than scanning. Compaction targets and query patterns have to move together, because a table that starts with range scans and later serves point lookups needs smaller row groups or a clustering key.

**Common mistake.** Setting an aggressive target size once and never revisiting it. File size is a tuning parameter tied to the dominant query shape, not a constant, and a streaming table's target should be smaller than an append-only fact table's.

## How do you detect and fix data skew in distributed processing?

Skew happens when some partitions have much more data than others, causing straggler tasks. Detect it via task duration distributions and partition size metrics. Fixes include salting keys, using skew-aware joins, repartitioning, or changing the join strategy (broadcast when possible).

Detect it in the Spark UI (one task taking 10x the median) and confirm by counting keys:

In [ ]:
# ILLUSTRATIVE - requires pyspark: install to run, or read on.
from pyspark.sql import functions as F

orders = spark.read.parquet("s3://acme-curated/orders/")
orders.groupBy("customer_id").count().orderBy(F.desc("count")).show(20, truncate=False)

Fix options, cheapest first: broadcast the small side; let Adaptive Query Execution handle skew joins; filter early; salt heavy keys; or use two-stage aggregation.

In [ ]:
# ILLUSTRATIVE - needs the live engine session and data this section names; read on for the pattern.
# Salting: spread a heavy key across N buckets, aggregate, then combine
SALT = 16

salt = (
    orders.withColumn("salt", (F.rand() * SALT).cast("int"))
    .withColumn("salted_key", F.concat_ws("#", "customer_id", "salt"))
)
partial = salt.groupBy("salted_key").agg(F.sum("amount").alias("partial_amount"))
final = (
    partial.withColumn("customer_id", F.split("salted_key", "#")[0])
    .groupBy("customer_id")
    .agg(F.sum("partial_amount").alias("amount"))
)
final.show()

In [ ]:
# ILLUSTRATIVE - needs the live engine session and data this section names; read on for the pattern.
# Prefer AQE skew handling when the join is the problem - often no code change
spark.conf.set("spark.sql.adaptive.enabled", "true")
spark.conf.set("spark.sql.adaptive.skewJoin.enabled", "true")
spark.conf.set("spark.sql.adaptive.skewJoin.skewedPartitionFactor", "5")

**Failure mode:** salting every key "just in case". It adds a shuffle stage and can slow an already balanced job. Measure first and prefer AQE unless you are on an older engine.

**Skew is a billing problem because the cluster stays up for the slowest task.** If one task holds 60 percent of a partition while the rest finish in a minute, you pay for idle executors plus the straggler. The waste is roughly `(wall_clock - mean_task_time) / wall_clock` of the stage cost, which is easy to see in the Spark UI task-duration chart.

In [ ]:
# Review only: what a straggler costs the stage
task_seconds = [58, 61, 59, 600, 60, 62]   # one skewed partition
stage_cost_per_second = 0.02
wall = max(task_seconds)
waste = (wall - sum(task_seconds) / len(task_seconds)) / wall
print(f"wall clock {wall}s, waste factor {waste:.1%}")
print(f"wasted spend this run {wall * stage_cost_per_second * waste:.2f}")

## What is shuffle and how do you reduce it in Spark?

Shuffle is data redistribution across executors (for joins, group by, distinct). It is expensive due to network IO and disk spills. You reduce shuffle by using proper partitioning, avoiding wide transformations, filtering early, broadcasting small tables, and tuning shuffle partitions.

The cost is roughly `data_size x (serialize + write + network + read)`, and spills multiply it. Ask whether the shuffle can be avoided entirely:

+ Replace `groupByKey` with `reduceByKey`/`agg` so partial aggregation happens first.
+ Broadcast a small dimension instead of shuffling the large side.
+ Filter and project **before** the join or group-by.
+ Bucket tables on the join key at write time so reads are shuffle-free.
+ Tune shuffle partitions for the actual data size.

In [ ]:
# ILLUSTRATIVE - requires pyspark: install to run, or read on.
from pyspark.sql import functions as F

orders = spark.read.parquet("s3://acme-curated/orders/")
customers = spark.read.parquet("s3://acme-dim/customers/")   # ~5 MB

slow = orders.join(customers, "customer_id")                 # full shuffle
fast = orders.join(F.broadcast(customers), "customer_id")    # no large shuffle

by_country = (
    orders.filter(F.col("order_date") >= "2026-10-01")
    .groupBy("country")
    .agg(F.sum("amount").alias("revenue"))
)

Then set `spark.sql.adaptive.enabled=true` and `spark.sql.adaptive.advisoryPartitionSizeInBytes=128m` so AQE coalesces shuffle partitions to the real data size instead of the default 200.

**Failure mode:** too many shuffle partitions. 10,000 partitions over 20 GB produces thousands of tiny tasks with scheduling and file-open overhead, which shows up in wall-clock and the cluster bill. AQE's partition coalescing usually fixes this automatically.

**Shuffle cost is bytes times a node-hour rate.** The engine serializes records, writes them to local disk, transfers them over the network, and reads them back, so the multiplier over raw data size is often 2-4x and a spill adds another pass. Reducing shuffle bytes, or pushing them onto faster local storage, moves the bill directly. Spill counters in the query history are the cheapest signal that shuffle is out of control.

```sql
-- Snowflake: queries that spilled to remote storage wasted shuffle work
SELECT query_id,
       ROUND(total_elapsed_time / 1000, 1) AS elapsed_s,
       ROUND(bytes_spilled_to_local_storage / POW(1024, 3), 2) AS local_spill_gb,
       ROUND(bytes_spilled_to_remote_storage / POW(1024, 3), 2) AS remote_spill_gb
FROM SNOWFLAKE.ACCOUNT_USAGE.QUERY_HISTORY
WHERE start_time >= DATEADD('day', -7, CURRENT_TIMESTAMP())
  AND bytes_spilled_to_remote_storage > 0
ORDER BY remote_spill_gb DESC
LIMIT 20;
```

## When should you pre-aggregate or materialize tables?

Materialize when many downstream queries reuse expensive computations or when interactive BI requires low latency. Avoid over-materialization because it increases storage and refresh complexity. A good approach is to materialize stable, high-value marts and keep exploratory logic as views.

Use this test: is the transformation **expensive**, **reused**, **stable**, and does it have a clear **grain and owner**? Three or more "yes" answers mean materialize; exploratory logic that changes weekly belongs in a view.

```sql
-- Snowflake: a dynamic table materializes and refreshes on its own
CREATE OR REPLACE DYNAMIC TABLE marts.revenue_daily
  TARGET_LAG = '1 hour'
  WAREHOUSE = transform_wh
AS
SELECT order_date, country, SUM(amount) AS revenue, COUNT(DISTINCT order_id) AS orders
FROM curated.fct_orders
GROUP BY order_date, country;
```

```yaml
# dbt: an incremental model only processes new partitions
# models/marts/revenue_daily.sql
# {{ config(materialized='incremental', unique_key='order_date',
#           partition_by={'field': 'order_date', 'data_type': 'date'},
#           cluster_by=['country']) }}
```

**Failure mode:** materializing everything "for speed". Each extra table adds storage, a refresh job, a failure mode, and a lineage edge. Watch the refresh fan-out too: 500 hourly incremental models can cost more in orchestration and small scans than one well-partitioned daily mart.

**The materialization breakeven is a count of reuses.** Let `R` be the cost of one refresh and `S` the scan saved per consumer query. The mart pays when `n x S > R`, so the real question is how many times a day the logic is reused. A mart refreshed hourly that serves 200 dashboard queries an hour clears that bar easily. One that serves a single weekly report usually does not.

```json
{
  "materialization_policy": [
    {"model": "marts.revenue_daily", "refresh": "hourly", "consumers": 14, "scan_saved_tib": 4.2},
    {"model": "marts.funnel_hourly", "refresh": "15m", "consumers": 2, "scan_saved_tib": 0.3,
     "review": "refresh cost may exceed saved scan"},
    {"model": "explore.session_paths", "refresh": "none", "consumers": 1, "review": "keep as a view"}
  ],
  "rule": "materialize when reuses_per_day * scan_saved > refresh_cost"
}
```

## How do you prevent runaway queries and protect shared clusters?

Use workload management: query timeouts, concurrency limits, resource quotas, and separate compute for heavy workloads. Enforce best practices with guardrails (linting, cost alerts) and educate users with profiling tools. Multi-tenant platforms typically need isolation to prevent one team from impacting others.

Layer controls from cheapest to strictest: budgets and alerts, byte limits, statement timeouts, concurrency caps, separate warehouses/pools, and query tagging for attribution.

```sql
-- Snowflake: suspend a warehouse at a credit threshold
CREATE OR REPLACE RESOURCE MONITOR limit_transform_wh WITH
  CREDIT_QUOTA = 500
  TRIGGERS ON 80 PERCENT DO NOTIFY
           ON 100 PERCENT DO SUSPEND
           ON 110 PERCENT DO SUSPEND_IMMEDIATE;

ALTER WAREHOUSE transform_wh SET RESOURCE_MONITOR = limit_transform_wh;
```

```bash
# CI guardrail: fail the pull request if a model would scan more than 500 GB
bq query --dry_run --use_legacy_sql=false --format=json < models/revenue.sql \
  | jq -e '.statistics.query.totalBytesProcessed < 500000000000'
```

**Failure mode:** quotas with no runbook. A suspended warehouse at 03:00 blocks the tier-1 pipeline, and on-call must know who to call and how to raise the limit. Pair every hard limit with an alert and an escalation path, and keep batch-critical workloads in a monitored window rather than competing with ad-hoc analysis.

**Guardrails are circuit breakers, not optimizers.** A resource monitor, byte limit, or concurrency cap does not make a workload cheaper; it stops the tail from becoming an incident. Set each threshold above normal p95 usage so ordinary work never trips it, and pair every hard limit with an alert, a runbook, and a named owner who can raise it. Quotas derived from the budget alone fail at month end, when the batch load is largest.

```json
{
  "budget": {"name": "analytics-monthly", "amount": 40000, "currency": "USD"},
  "alerts": [
    {"threshold_percent": 50, "notify": ["data-platform@acme.com"]},
    {"threshold_percent": 80, "notify": ["data-platform@acme.com", "finops@acme.com"]},
    {"threshold_percent": 100, "notify": ["finance@acme.com"],
     "action": "require_approval_for_new_warehouses"}
  ],
  "anomaly_detection": {"lookback_days": 30, "sensitivity": "medium",
                        "notify": ["oncall-data@acme.com"]}
}
```

## How do you optimize joins in large-scale analytics?

You optimize joins by ensuring join keys are clean and well-distributed, filtering before joins, and choosing appropriate join strategies (broadcast vs shuffle). You also reduce the size of join inputs (select only needed columns) and consider pre-joining into curated marts when joins are repeated.

| Join shape | Strategy | Why |
|---|---|---|
| Huge x tiny (< 10 MB) | broadcast | no shuffle of the large side |
| Huge x medium | broadcast if safe, else bucket | avoids the shuffle |
| Huge x huge, filterable | filter/project, then shuffle | moves less data |
| Huge x huge, repeated daily | pre-join into a mart | pay once, reuse |
| Skewed keys | AQE skew join or salting | avoids stragglers |

```sql
-- Reduce both sides before the join: project, filter, then join
WITH recent_orders AS (
    SELECT order_id, customer_id, amount, order_date
    FROM curated.fct_orders
    WHERE order_date >= DATE '2026-10-01'
),
active_customers AS (
    SELECT customer_id, country, segment
    FROM curated.dim_customers
    WHERE is_active = TRUE
)
SELECT c.country, c.segment, SUM(o.amount) AS revenue
FROM recent_orders AS o
JOIN active_customers AS c USING (customer_id)
GROUP BY c.country, c.segment;
```

In [ ]:
# ILLUSTRATIVE - requires pyspark: install to run, or read on.
# PySpark: project and filter before broadcasting the dimension
from pyspark.sql import functions as F

orders = spark.read.parquet("s3://acme-curated/orders/").select(
    "order_id", "customer_id", "amount", "order_date"
)
customers = spark.read.parquet("s3://acme-dim/customers/").select(
    "customer_id", "country", "segment"
)

result = (
    orders.filter(F.col("order_date") >= "2026-10-01")
    .join(F.broadcast(customers), "customer_id")
    .groupBy("country", "segment")
    .agg(F.sum("amount").alias("revenue"))
)

**Failure mode:** broadcast joins that outgrow memory. If the "small" side grows past the broadcast threshold, the driver can OOM. Set `spark.sql.autoBroadcastJoinThreshold` deliberately and monitor it rather than assuming a dimension stays small forever.

**Join cost is build-side size times probe rows, plus shuffle.** A broadcast join removes the shuffle of the large table but copies the small table to every executor, so it wins only while the small side is genuinely small and memory headroom exists. Bucketing pays the sort cost once at write time and removes it on every read, which makes repeated joins on the same key the best bucketing candidates. Filtering before the join reduces both sides at once.

```sql
-- BigQuery: slot-heavy queries are usually the join and shuffle heavy ones
SELECT job_id, user_email,
       ROUND(total_slot_ms / 1000 / 3600, 2) AS slot_hours,
       ROUND(total_bytes_billed / POW(1024, 3), 1) AS gb_billed,
       SUBSTR(query, 1, 120) AS query_head
FROM `region-us`.INFORMATION_SCHEMA.JOBS_BY_PROJECT
WHERE creation_time >= TIMESTAMP_SUB(CURRENT_TIMESTAMP(), INTERVAL 7 DAY)
  AND job_type = 'QUERY' AND state = 'DONE'
ORDER BY total_slot_ms DESC
LIMIT 20;
```

## How do you plan and estimate the cost of a backfill?

Estimate based on data volume, compute requirements, and expected scan/shuffle behavior. Run on a small sample window to measure throughput and extrapolate. Backfills should be staged, monitored, and preferably run in off-peak windows, with a clear rollback plan.

A defensible estimate has four numbers: bytes scanned, shuffle bytes, cluster/warehouse hours, and wall-clock. Measure one representative partition, then multiply.

```sql
-- Cost of rebuilding one day: bytes and slot time, then extrapolate
SELECT total_bytes_billed / POW(1024, 4) AS tib_billed,
       total_slot_ms / 1000 / 3600 AS slot_hours
FROM `region-us`.INFORMATION_SCHEMA.JOBS_BY_PROJECT
WHERE job_id = 'backfill_orders_2026_09_01' AND state = 'DONE';
```

In [ ]:
# 1 day measured -> 90 days planned, with a safety margin
per_day_bytes = 1.2 * 1024 ** 4
per_day_slot_hours = 2.5
days, margin = 90, 1.3

print("scan TiB:", round(per_day_bytes * days / 1024 ** 4, 1))
print("slot-hours:", round(per_day_slot_hours * days * margin, 1))

```bash
# Off-peak backfill with bounded concurrency
airflow dags backfill revenue_daily \
  --start-date 2026-07-01 --end-date 2026-09-30 \
  --max-active-runs 2 --reset-dagruns
```

Controls that keep a backfill from becoming an incident: run it on a **separate warehouse/cluster**, cap concurrency, checkpoint progress per partition, and stop if cumulative spend crosses a threshold. Publish the estimate and the expected metric change to stakeholders beforehand.

**Spot and committed capacity change the rate, not the work.** Spot and preemptible nodes cut the hourly rate by 60-90 percent but can be reclaimed, so they suit restartable, checkpointed batch jobs and not interactive or stateful ones. Reserved instances, savings plans, and capacity commitments trade a one- or three-year term for a 20-50 percent discount and only pay off above a stable baseline. Commit to the floor you always run and leave the peak on demand.

**Common mistake.** Committing to average usage instead of the floor. The average includes peaks you would have paid for on demand anyway, so the commitment delivers credits you never use.

```sql
-- Snowflake: hourly baseline versus peak, used to size a commitment
SELECT DATE_TRUNC('hour', start_time) AS hour,
       SUM(credits_used) AS credits
FROM SNOWFLAKE.ACCOUNT_USAGE.WAREHOUSE_METERING_HISTORY
WHERE start_time >= DATEADD('day', -90, CURRENT_TIMESTAMP())
GROUP BY hour
ORDER BY credits ASC;
```

## What metrics would you track for FinOps in data engineering?

Common metrics include cost by team/project, cost per pipeline, bytes scanned per query, cluster utilization, storage growth by layer, and top expensive datasets/queries. You also track trend changes (week-over-week) and tie cost to value (usage, criticality).

A useful dashboard has four rows: **spend** (by warehouse/project, with budget burn), **efficiency** (cost per dashboard refresh, per 1,000 queries), **waste** (idle warehouse time, failed-job spend, time-travel storage, unqueried tables), and **value** (spend versus query volume by team).

```sql
-- Cost per pipeline, attributed by query tag (Snowflake)
SELECT REGEXP_SUBSTR(query_tag, 'pipeline=([^;]+)', 1, 1, 'e', 1) AS pipeline,
       COUNT(*) AS queries,
       ROUND(SUM(bytes_scanned) / POW(1024, 4), 2) AS tib_scanned
FROM SNOWFLAKE.ACCOUNT_USAGE.QUERY_HISTORY
WHERE start_time >= DATEADD('day', -7, CURRENT_TIMESTAMP())
  AND query_tag ILIKE '%pipeline=%'
GROUP BY pipeline
ORDER BY tib_scanned DESC
LIMIT 25;
```

```sql
-- Waste: storage for tables not queried in 90 days
SELECT t.table_schema, t.table_name,
       ROUND(t.active_bytes / POW(1024, 3), 1) AS active_gb,
       MAX(q.start_time) AS last_queried
FROM SNOWFLAKE.ACCOUNT_USAGE.TABLE_STORAGE_METRICS AS t
LEFT JOIN SNOWFLAKE.ACCOUNT_USAGE.QUERY_HISTORY AS q
  ON q.query_text ILIKE '%' || t.table_name || '%'
GROUP BY t.table_schema, t.table_name, t.active_bytes
HAVING last_queried IS NULL OR last_queried < DATEADD('day', -90, CURRENT_TIMESTAMP())
ORDER BY active_gb DESC;
```

Pair every metric with a **target and an owner** ("p95 bytes scanned per dashboard query < 50 GB, owned by BI platform"). Without a target, a FinOps dashboard is just a second bill. Review cost in the same ritual as reliability and treat a sustained budget breach as an incident with a root cause.

## How do you cut cost without hurting reliability?

Cost work fails in two directions: leaving obvious waste in place, or cutting so hard that an SLA, a recovery path, or an audit trail disappears. The reliable way to cut is to remove work nobody needs, keep the work the business depends on, and make the trade explicit with numbers. The four sections below move from finding the drivers, to ranking the levers, to the guardrails that keep a cut safe, to the operating rhythm that stops spend from drifting back.

A cut is safe when it changes none of correctness, freshness, recovery time, or peak concurrency. Anything that touches one of those is a reliability decision as much as a cost decision, and it needs an owner who has read the SLA. Cost work that ignores that boundary gets reverted, and the revert costs more than the saving.

## How do you find the top cost drivers?

Start from the billing system tables rather than the invoice. On Databricks, `system.billing.usage` records usage quantities by SKU and nests the resource ids under `usage_metadata`, while `system.billing.list_prices` holds the historical list price per SKU. Join them on `sku_name` and the effective price window to turn DBUs into currency, then group by workload. Corrections arrive as `RETRACTION` rows with negative quantities, so keep `record_type = 'ORIGINAL'` for a simple view or sum all record types for the corrected figure.

```sql
-- Databricks: cost by SKU and workload, joining usage to list prices
WITH priced AS (
  SELECT u.usage_date,
         u.sku_name,
         u.billing_origin_product,
         u.usage_metadata.job_id       AS job_id,
         u.usage_metadata.warehouse_id AS warehouse_id,
         u.usage_metadata.cluster_id   AS cluster_id,
         u.identity_metadata.run_as    AS run_as,
         u.usage_quantity,
         p.pricing.effective_list.default AS list_price
  FROM system.billing.usage AS u
  JOIN system.billing.list_prices AS p
    ON p.sku_name = u.sku_name
   AND u.usage_start_time >= p.price_start_time
   AND (u.usage_start_time < p.price_end_time OR p.price_end_time IS NULL)
  WHERE u.usage_date >= CURRENT_DATE() - INTERVAL 30 DAYS
    AND u.record_type = 'ORIGINAL'
)
SELECT sku_name,
       COALESCE(job_id, warehouse_id, cluster_id) AS workload_id,
       ROUND(SUM(usage_quantity * list_price), 2) AS cost_usd
FROM priced
GROUP BY sku_name, COALESCE(job_id, warehouse_id, cluster_id)
ORDER BY cost_usd DESC
LIMIT 25;
```

```sql
-- Athena over the Cost and Usage Report: spend by service and usage type
SELECT line_item_product_code                  AS service,
       line_item_usage_type                    AS usage_type,
       ROUND(SUM(line_item_unblended_cost), 2) AS cost_usd
FROM cur.acme_billing
WHERE line_item_usage_start_date >= date_trunc('month', current_date - interval '2' month)
  AND line_item_line_item_type IN ('Usage', 'DiscountedUsage', 'SavingsPlanCoveredUsage')
GROUP BY 1, 2
ORDER BY cost_usd DESC
LIMIT 50;
```

**Turn the totals into unit costs.** Divide spend by something the business cares about: cost per 1,000 pipeline runs, per 1,000 dashboard queries, per active customer, or per GB delivered. Unit costs are comparable across months and expose regressions that a flat total hides. Publish a small scorecard rather than a raw bill, and slice it by team so the people who can change a workload can see their own number.

## Which cost levers give the biggest savings?

Rank levers by savings and by risk, and take the reversible configuration changes first. The table is the usual ordering by expected saving for a platform that has never been optimized.

| Lever | Typical saving | Risk |
|---|---|---|
| Auto-suspend and right-sizing | 20-60% of compute | low; watch queue time at peak |
| Lifecycle tiering and retention | 30-70% of storage | low; retrieval latency and early-delete fees |
| Compaction and file layout | 10-40% of scan cost | low if transactional, medium if hand-written |
| Scheduling and off-peak placement | 5-25% of compute | low; freshness windows shift |
| Spot or preemptible capacity | 60-90% of the node rate | medium; preemption needs checkpoints |
| Commitments and reservations | 20-50% of the baseline rate | medium; over-commitment wastes credits |

**Sequence matters.** Configuration changes (auto-suspend, retention, schedule) are reversible and land first. Layout changes (partitioning, clustering, compaction) take a rewrite, so they follow a measurement and a backfill estimate. Rate changes (spot, commitments) are contracts, so they come last, once the workload shape is stable and the baseline is known. Changing the rate before the shape locks in waste for the length of the contract.

## How do you set guardrails and budgets for cost?

Guardrails sit at the platform boundary, not inside every pipeline. A budget tracks cumulative spend and fires alerts at thresholds. A resource monitor or quota stops a runaway warehouse before it consumes a month of credits. An anomaly alert catches the step changes that a monthly budget would only reveal after the fact. An approval workflow adds friction to the few actions that create large, durable spend, such as a new always-on warehouse or a large commitment.

```yaml
# Budget and tag policy, enforced at the platform boundary
budgets:
  - name: analytics-monthly
    amount: 40000
    scope: workspace:analytics
    alerts:
      - at: 50%
        notify: [data-platform@acme.com]
      - at: 90%
        notify: [data-platform@acme.com, finops@acme.com]
      - at: 100%
        notify: [finance@acme.com]
        action: notify_and_require_approval
required_tags:
  - cost_center
  - team
  - pipeline
  - env
enforcement:
  untagged_workloads: deny_new_warehouse
  tag_coverage_floor: 0.90
```

```json
{
  "anomaly_alert": {
    "metric": "daily_spend",
    "scope": "workspace",
    "window_days": 30,
    "method": "trailing_median",
    "trigger": "value > max(1.5 * median, median + 500)",
    "notify": ["oncall-data@acme.com"]
  }
}
```

**Common mistake.** Setting quotas below normal peak so batch jobs fail at month end. Derive thresholds from observed p95 usage, alert well before the hard stop, and keep the runbook next to the alert rather than in a wiki nobody opens at 03:00.

## What is the cost optimization operating rhythm?

**Weekly review, monthly deep dive, quarterly commitment check.** The weekly review reads the anomaly list and the top movers, assigns an owner to each, and closes last week's actions. The monthly deep dive picks one structural lever, quantifies it, and ships it. The quarterly check re-sizes commitments and retires datasets nobody queries. The rhythm matters more than any single optimization, because spend drifts back within a quarter if nobody owns it.

A short KPI list is enough:

+ cost per pipeline run and its week-over-week change
+ cost per 1,000 dashboard queries
+ unallocated spend as a percentage of total
+ idle compute minutes as a percentage of billed compute
+ storage by tier and by age band
+ open cost actions with owners and due dates

Every KPI needs a target and an owner, for example "p95 bytes scanned per dashboard query under 50 GB, owned by the BI platform team". Treat a sustained budget breach as an incident with a root cause, not as a monthly surprise.

## Cost optimization exercises

These exercises use only the Python standard library and model the cost arithmetic behind the levers in this notebook. Each one names the task, gives a hint, states the expected result, and then runs a Python cell with asserts. Run them with the repository virtual environment: `/Users/sai/WS/data-engineering-interview-questions/.venv/bin/python`.

The numbers are models, not predictions. Treat the asserts as a check on the formula, and the value of each exercise as knowing which quantity to compute and which direction a change moves it.

### Try it 1. Cost per pipeline run

**Task.** A pipeline runs 40 times a day. Each run uses a 6 DBU/hour cluster for 15 minutes plus 0.5 DBU of serverless tasks. At 0.50 per DBU, compute the cost per run and the 30-day cost.

**Hint.** Cluster DBUs are `dbu_per_hour x minutes / 60`; add the serverless DBUs, then multiply by the price.

**Expected result.** `1.00` per run and `1200.00` for 30 days.

In [ ]:
DBU_PER_HOUR = 6.0
RUN_MINUTES = 15
SERVERLESS_DBU_PER_RUN = 0.5
PRICE_PER_DBU = 0.50
RUNS_PER_DAY = 40
DAYS = 30


def cost_per_run():
    cluster_dbu = DBU_PER_HOUR * RUN_MINUTES / 60
    return (cluster_dbu + SERVERLESS_DBU_PER_RUN) * PRICE_PER_DBU


per_run = cost_per_run()
monthly = per_run * RUNS_PER_DAY * DAYS
print(f"cost per run = {per_run:.2f}")
print(f"monthly = {monthly:.2f}")

assert round(per_run, 2) == 1.00
assert round(monthly, 2) == 1200.00

### Try it 2. Savings from lifecycle tiering

**Task.** A 200 TB storage profile has 80 TB hot, 60 TB infrequent, and 60 TB archived. At 0.023, 0.0125, and 0.004 per GB-month, compute the tiered monthly cost, the all-hot cost, and the saving percentage.

**Hint.** Multiply TB by 1024 to get GB, then by the rate. The saving percentage is `(all_hot - tiered) / all_hot`.

**Expected result.** Tiered `2897.92`, all-hot `4710.40`, saving `38.5%`.

In [ ]:
HOT_RATE = 0.023
INFREQUENT_RATE = 0.0125
ARCHIVE_RATE = 0.004
GB_PER_TB = 1024
profile = {"hot_tb": 80, "infrequent_tb": 60, "archive_tb": 60}


def monthly_cost(tb, rate):
    return tb * GB_PER_TB * rate


tiered = (monthly_cost(profile["hot_tb"], HOT_RATE)
          + monthly_cost(profile["infrequent_tb"], INFREQUENT_RATE)
          + monthly_cost(profile["archive_tb"], ARCHIVE_RATE))
all_hot = monthly_cost(sum(profile.values()), HOT_RATE)
saving_pct = (all_hot - tiered) / all_hot
print(f"tiered = {tiered:.2f}")
print(f"all hot = {all_hot:.2f}")
print(f"saving = {all_hot - tiered:.2f} ({saving_pct:.1%})")

assert round(tiered, 2) == 2897.92
assert round(all_hot, 2) == 4710.40
assert round(saving_pct * 100, 1) == 38.5

### Try it 3. Auto-suspend savings from idle hours

**Task.** An XL warehouse burns 16 credits per hour and currently runs 24 hours a day. With auto-suspend it is active 6 hours a day, with 8 starts that each add 2 minutes of idle tail. Compute the monthly credits before and after, and the saving percentage.

**Hint.** Convert the idle tail to hours (`starts x minutes / 60`) and add it to the active hours before multiplying by the rate and 30 days.

**Expected result.** `11520.0` credits before, `3008.0` after, a `73.9%` saving.

In [ ]:
CREDITS_PER_HOUR = 16          # XL warehouse
ALWAYS_ON_HOURS = 24
ACTIVE_HOURS = 6
STARTS_PER_DAY = 8
IDLE_MINUTES_PER_START = 2
DAYS = 30

always_on_daily = ALWAYS_ON_HOURS * CREDITS_PER_HOUR
idle_hours = STARTS_PER_DAY * IDLE_MINUTES_PER_START / 60
suspended_daily = (ACTIVE_HOURS + idle_hours) * CREDITS_PER_HOUR
monthly_before = always_on_daily * DAYS
monthly_after = suspended_daily * DAYS
saving_pct = (monthly_before - monthly_after) / monthly_before
print(f"always on:    {monthly_before:.1f} credits/month")
print(f"auto-suspend: {monthly_after:.1f} credits/month")
print(f"saving: {monthly_before - monthly_after:.1f} credits ({saving_pct:.1%})")

assert round(monthly_before, 1) == 11520.0
assert round(monthly_after, 1) == 3008.0
assert round(saving_pct * 100, 1) == 73.9

### Try it 4. Compaction savings from small files

**Task.** A partition holds 400,000 files of 4 MB. Compacting to a 256 MB target costs 8 ms of planning per file. Compute the new file count, the file reduction, and the planning hours saved per day across 60 scans.

**Hint.** New file count is `ceil(total_MB / target_MB)`. Planning seconds are `files x ms / 1000`, and the daily saving multiplies by scans.

**Expected result.** `6250` files, `98.4%` fewer, `52.5` planning hours saved per day.

In [ ]:
import math

FILES = 400_000
AVG_MB = 4
TARGET_MB = 256
PLAN_MS_PER_FILE = 8
SCANS_PER_DAY = 60
REWRITE_COST_PER_GB = 0.005

total_mb = FILES * AVG_MB
new_files = math.ceil(total_mb / TARGET_MB)
reduction = 1 - new_files / FILES
plan_before_s = FILES * PLAN_MS_PER_FILE / 1000
plan_after_s = new_files * PLAN_MS_PER_FILE / 1000
daily_plan_hours = (plan_before_s - plan_after_s) * SCANS_PER_DAY / 3600
rewrite_cost = total_mb / 1024 * REWRITE_COST_PER_GB
print(f"files: {FILES:,} -> {new_files:,} ({reduction:.1%} fewer)")
print(f"planning per scan: {plan_before_s:.0f}s -> {plan_after_s:.0f}s")
print(f"planning saved: {daily_plan_hours:.1f} cluster-hours/day")
print(f"one-time rewrite cost: {rewrite_cost:.2f}")

assert new_files == 6250
assert round(reduction * 100, 1) == 98.4
assert round(daily_plan_hours, 1) == 52.5

### Try it 5. Spot versus on-demand break-even

**Task.** A job needs 500 node-hours. On-demand costs 0.60 per node-hour and spot costs 0.42. A preemption happens about every 10 node-hours and loses 2 node-hours of work that must rerun. Compute the spot cost, the saving, and the rework fraction at which spot stops winning.

**Hint.** Spot hours are `job_hours x (1 + rework_hours / hours_between_preemptions)`. Break-even rework is `on_demand / spot - 1`.

**Expected result.** Spot `252.00`, a `16.0%` saving, break-even rework `42.9%`.

In [ ]:
ON_DEMAND = 0.60
SPOT = 0.42
JOB_HOURS = 500
REWORK_HOURS_PER_PREEMPTION = 2
HOURS_BETWEEN_PREEMPTIONS = 10

rework_fraction = REWORK_HOURS_PER_PREEMPTION / HOURS_BETWEEN_PREEMPTIONS
spot_hours = JOB_HOURS * (1 + rework_fraction)
on_demand_cost = JOB_HOURS * ON_DEMAND
spot_cost = spot_hours * SPOT
saving_pct = (on_demand_cost - spot_cost) / on_demand_cost
break_even_rework = ON_DEMAND / SPOT - 1
print(f"on-demand: {on_demand_cost:.2f}")
print(f"spot:      {spot_cost:.2f} ({saving_pct:.1%} cheaper)")
print(f"spot stops winning when rework exceeds {break_even_rework:.1%} of hours")

assert round(rework_fraction, 2) == 0.20
assert round(spot_cost, 2) == 252.00
assert round(saving_pct * 100, 1) == 16.0
assert round(break_even_rework * 100, 1) == 42.9

### Try it 6. Commitment break-even from baseline usage

**Task.** A day has 16 busy hours at 40 credits and 8 quiet hours at 5 credits, for 30 days. A one-year commitment gives a 30% discount at the floor of 5 credits per hour, and anything above the floor is bought on demand. Compute on-demand credits, committed credits, the saving, and the break-even utilization.

**Hint.** Per hour you pay `commit x (1 - discount)` plus `max(0, usage - commit)`. Break-even utilization is `1 - discount`, because below it you pay for credits you never use.

**Expected result.** On-demand `20400`, committed `19320`, saving `1080`, break-even utilization `70%`.

In [ ]:
PRICE_PER_CREDIT = 3.00
DISCOUNT = 0.30
DAY_HOURS = 24
DAYS = 30
BUSY_HOURS, BUSY_CREDITS = 16, 40
QUIET_HOURS, QUIET_CREDITS = 8, 5

usage = [BUSY_CREDITS] * BUSY_HOURS + [QUIET_CREDITS] * QUIET_HOURS
on_demand_credits = sum(usage) * DAYS
commit = min(usage)
committed_credits = sum(commit * (1 - DISCOUNT) + max(0, u - commit) for u in usage) * DAYS
saving = on_demand_credits - committed_credits
break_even_utilization = 1 - DISCOUNT
print(f"on-demand: {on_demand_credits:.0f} credits")
print(f"committed: {committed_credits:.0f} credits")
print(f"saving: {saving:.0f} credits ({saving / on_demand_credits:.1%})")
print(f"break-even utilization: {break_even_utilization:.0%}")

assert round(on_demand_credits) == 20400
assert round(committed_credits) == 19320
assert round(saving) == 1080
assert round(break_even_utilization, 2) == 0.70

### Try it 7. Tag coverage versus unallocated spend

**Task.** Given tagged and untagged spend for four teams, compute total coverage, unallocated spend, and the coverage after fully tagging the team with the worst untagged share.

**Hint.** Coverage is `tagged / (tagged + untagged)`. The worst team is the one with the largest `untagged / (tagged + untagged)` ratio.

**Expected result.** Coverage `79.6%`, unallocated `9,500`, and `91.4%` after fixing the worst team.

In [ ]:
rows = [
    ("bi", 12000, 500),
    ("ml", 8000, 3000),
    ("etl", 15000, 0),
    ("adhoc", 2000, 6000),
]


def totals(data):
    tagged = sum(t for _, t, _ in data)
    untagged = sum(u for _, _, u in data)
    return tagged, untagged, tagged + untagged


tagged, untagged, total = totals(rows)
coverage = tagged / total
worst = max(rows, key=lambda r: r[2] / (r[1] + r[2]))
fixed = [(name, t, 0) if name == worst[0] else (name, t, u) for name, t, u in rows]
fixed_coverage = totals(fixed)[0] / totals(fixed)[2]
print(f"tagged {tagged:,} of {total:,} -> coverage {coverage:.1%}")
print(f"unallocated spend {untagged:,}")
print(f"fully tagging {worst[0]} raises coverage to {fixed_coverage:.1%}")

assert round(coverage * 100, 1) == 79.6
assert untagged == 9500
assert round(fixed_coverage * 100, 1) == 91.4

### Try it 8. Anomaly detection from a daily spend series

**Task.** Given 12 baseline days of spend and 4 later days, flag anomalies using a threshold of `max(1.5 x median, median + 500)`.

**Hint.** Use `statistics.median` on the baseline, compute the threshold once, and flag any later day strictly above it.

**Expected result.** Median `1017.5`, threshold `1526.25`, and anomalies `[3400, 1600]`.

In [ ]:
import statistics

baseline = [1000, 1050, 980, 1020, 1100, 990, 1010, 1030, 1000, 1060, 1020, 1015]
observed = [1040, 3400, 1450, 1600]

median = statistics.median(baseline)
threshold = max(1.5 * median, median + 500)
flagged = [value for value in observed if value > threshold]
print(f"median daily spend = {median:.1f}")
print(f"alert threshold = {threshold:.2f}")
print(f"anomalies = {flagged}")

assert median == 1017.5
assert round(threshold, 2) == 1526.25
assert flagged == [3400, 1600]

## Real world use cases for Cost Optimization

The scenarios below put the levers together end to end. Each one names the workload, the change, and the measured or expected effect, and each shows why the architectural decision mattered more than the individual tuning.

### 1. E-commerce: cutting a BigQuery bill with partitioning and clustering

A retailer's on-demand BigQuery bill reached ~$68k/month because the top dashboards scanned raw event tables every hour. Daily event volume is ~2 TB and the fact table holds 300 TB.

```sql
-- After: partition by day, cluster by country; the same query scans ~1/50th
CREATE OR REPLACE TABLE curated.events
PARTITION BY DATE(event_ts)
CLUSTER BY country, event_type
AS SELECT * FROM raw.events_json;

SELECT country, SUM(amount) AS revenue
FROM curated.events
WHERE event_ts >= TIMESTAMP_SUB(CURRENT_TIMESTAMP(), INTERVAL 30 DAY)
GROUP BY country;
```

**Why this design:** on-demand pricing is linear in bytes scanned, so partition pruning plus clustering was the highest-leverage change and needed no query rewrites beyond removing the `DATE()` wrapper. A CI dry-run gate stops the regression from returning.

### 2. Snowflake: right-sizing warehouses and killing idle time

A SaaS company ran a single XL warehouse 24/7 for all workloads; ETL and analyst queries contended, so both were slow and the warehouse rarely suspended.

```sql
CREATE WAREHOUSE etl_wh   WAREHOUSE_SIZE = LARGE  AUTO_SUSPEND = 60  AUTO_RESUME = TRUE;
CREATE WAREHOUSE bi_wh    WAREHOUSE_SIZE = MEDIUM AUTO_SUSPEND = 300 AUTO_RESUME = TRUE;
CREATE WAREHOUSE adhoc_wh WAREHOUSE_SIZE = XSMALL AUTO_SUSPEND = 60  AUTO_RESUME = TRUE;

ALTER WAREHOUSE bi_wh SET MIN_CLUSTER_COUNT = 1 MAX_CLUSTER_COUNT = 4 SCALING_POLICY = 'ECONOMY';
```

**Why this design:** separating ETL from BI removed contention, let each warehouse match its workload, and aggressive auto-suspend eliminated idle hours. Multi-cluster BI keeps analyst latency predictable while billing only for clusters actually used.

### 3. Fintech: controlling a Spark backfill that would have cost a month of budget

A payments team needed to reprocess 18 months of transactions (~180 TB) after a currency-conversion bug. A naive full reprocess on on-demand clusters would have cost tens of thousands of dollars.

In [ ]:
# ILLUSTRATIVE - needs the live engine session and data this section names; read on for the pattern.
# Spot instances, dynamic allocation, and per-month checkpoints
spark = (
    SparkSession.builder.appName("fx-reprocess-2026")
    .config("spark.executor.instances", "40")
    .config("spark.dynamicAllocation.enabled", "true")
    .config("spark.sql.adaptive.enabled", "true")
    .getOrCreate()
)

for month in month_range("2025-01", "2026-06"):
    if checkpoint_done(month):
        continue                                   # restartable: skip completed months
    (spark.read.parquet(f"s3://acme-raw/txn/month={month}/")
          .transform(convert_currency)
          .write.mode("overwrite")
          .parquet(f"s3://acme-curated/txn/month={month}/"))
    checkpoint_mark(month)

**Why this design:** spot capacity with dynamic allocation cut the compute rate, per-month checkpoints meant a preemption lost at most one month, and a one-month pilot validated the per-month cost before committing to all 18.

### 4. Multi-tenant SaaS: taming streaming small files

A platform ingests 5 TB/day via Kafka into S3 using 30-second micro-batches, producing ~300,000 small Parquet files per day. Query planning dominated runtime and S3 request charges dominated storage cost.

In [ ]:
# ILLUSTRATIVE - needs the live engine session and data this section names; read on for the pattern.
# Longer trigger interval reduces file creation at the source
(
    spark.readStream.format("kafka")
    .option("kafka.bootstrap.servers", "broker:9092")
    .option("subscribe", "events")
    .load()
    .writeStream
    .format("iceberg")
    .option("checkpointLocation", "s3://acme-checkpoints/events/")
    .trigger(processingTime="10 minutes")
    .toTable("lake.events")
)

```sql
-- Then compact history transactionally and expire old snapshots
CALL lake.system.rewrite_data_files(
    table => 'lake.events',
    options => map('target-file-size-bytes', '268435456', 'min-input-files', '5')
);
CALL lake.system.expire_snapshots(table => 'lake.events', older_than => TIMESTAMP '2026-10-01 00:00:00');
```

**Why this design:** compaction alone would have run forever while the writer kept producing tiny files, so the pipeline first increased the commit interval and then used Iceberg's transactional compaction and snapshot expiry. Expiring snapshots also cut metastore size, which had become a planning bottleneck.